# 🌊 Global Solution 2026 — FIAP
## Monitoramento de Riscos Ambientais — Análise Interativa
**Cenário A: Rede de Resposta a Enchentes no Rio Grande do Sul**

> **Como usar este notebook:** Execute as células em ordem (Shift+Enter). 
> As células marcadas com `⚙️ PARÂMETRO` são onde você pode **alterar os valores** para explorar diferentes cenários.

---

In [ ]:
# Configuração do ambiente
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data_structures import Grafo, BinarySearchTree, criar_vertice
from src.data_structures import id_v, nome_v, risco_v, custo_v, pop_v
from src.dataset_rs import construir_dataset_rs
from src.brute_force import ForcaBruta, contar_caminhos_por_n
from src.greedy import Dijkstra, Prim
from src.performance_monitor import executar_benchmarks, gerar_grafo_sintetico
from src.visualizations import (
    fig1_grafo_mst, fig2_bst, fig3_desempenho, fig4_gap, fig5_tabela_estruturas
)

import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline

print('✅ Ambiente configurado!')

## 1. Construção do Grafo e da BST


In [ ]:
grafo, bst, vertices = construir_dataset_rs()
print(grafo)
print(bst)
print(f'\nTotal de municípios: {grafo.num_vertices()}')
print(f'Total de conexões:   {grafo.num_arestas()}')

## 2. ⚙️ Exploração da BST — Altere os parâmetros abaixo


In [ ]:
# ⚙️ PARÂMETRO — mude estes valores para filtrar por faixa de risco
RISCO_MIN = 0.70   # mínimo: 0.0
RISCO_MAX = 1.00   # máximo: 1.0

resultado = bst.buscar(RISCO_MIN, RISCO_MAX)
print(f'Municípios com risco entre {RISCO_MIN:.2f} e {RISCO_MAX:.2f}:')
print(f'{"Nome":<25} {"Risco":>6} {"Pop":>10} {"Custo(R$mil)":>14}')
print('-' * 60)
for v in sorted(resultado, key=lambda x: -risco_v(x)):
    print(f'{nome_v(v):<25} {risco_v(v):>6.2f} {pop_v(v):>10,} {custo_v(v):>14,.0f}')
print(f'\nTotal: {len(resultado)} municípios encontrados')

## 3. ⚙️ Dijkstra — Rota de Atendimento Prioritário


In [ ]:
# ⚙️ PARÂMETRO — mude o hub de origem e o limiar de risco
# IDs disponíveis: 4314902=Porto Alegre, 4307005=Canoas, 4304200=Caxias do Sul
ID_ORIGEM      = 4314902   # hub de recursos (Porto Alegre por padrão)
LIMIAR_RISCO   = 0.65      # só municípios com risco >= este valor aparecem na agenda

dijk = Dijkstra(grafo, bst)
agenda, res = dijk.rota_prioritaria(ID_ORIGEM, limiar_risco=LIMIAR_RISCO)

origem_nome = grafo.get_vertice(ID_ORIGEM)[1]
print(f'Hub de origem: {origem_nome}')
print(f'Municípios críticos (risco ≥ {LIMIAR_RISCO}): {len(agenda)}\n')
print(f'{"Município":<25} {"Risco":>6} {"Dist(h)":>8}  Rota')
print('-' * 80)
for item in agenda:
    rota = ' → '.join(item['caminho_nomes'])
    dist = f"{item['custo_h']:.2f}" if item['custo_h'] < float('inf') else '∞'
    print(f"{item['nome']:<25} {item['risco']:>6.2f} {dist:>8}  {rota}")

print(f'\n📊 Arestas relaxadas: {res.arestas_relaxadas} | Tempo: {res.tempo_ms:.3f} ms')

## 4. ⚙️ Força Bruta — Caminho entre dois municípios


In [ ]:
# ⚙️ PARÂMETRO — escolha origem e destino (use IDs do grafo)
# Dica: IDs disponíveis podem ser vistos com: print(list(grafo.todos_ids()))
ID_ORIGEM_FB  = 4314902  # Porto Alegre
ID_DESTINO_FB = 4313375  # Muçum

print(list(grafo.todos_ids()))  # mostra todos os IDs disponíveis

fb = ForcaBruta(grafo)
res_fb = fb.encontrar_caminho_minimo(ID_ORIGEM_FB, ID_DESTINO_FB)

orig_nome = grafo.get_vertice(ID_ORIGEM_FB)[1]
dest_nome = grafo.get_vertice(ID_DESTINO_FB)[1]

if res_fb.custo_otimo < float('inf'):
    rota = ' → '.join(grafo.get_vertice(v)[1] for v in res_fb.caminho_otimo)
    print(f'\nOrigem:  {orig_nome}')
    print(f'Destino: {dest_nome}')
    print(f'\n✅ Custo ótimo: {res_fb.custo_otimo:.3f} h')
    print(f'📍 Rota: {rota}')
else:
    print(f'⚠️ Não há caminho de {orig_nome} até {dest_nome}')

print(f'\n🔁 Chamadas recursivas:  {res_fb.num_chamadas_rec:,}')
print(f'🛣️  Caminhos avaliados:   {res_fb.num_caminhos_avaliados:,}')
print(f'⏱️  Tempo:                {res_fb.tempo_ms:.4f} ms')

# Validação: compara com Dijkstra
dijk2 = Dijkstra(grafo)
res_dijk2 = dijk2.executar(ID_ORIGEM_FB)
custo_dijk = res_dijk2.distancias.get(ID_DESTINO_FB, float('inf'))
if res_fb.custo_otimo < float('inf') and custo_dijk < float('inf'):
    gap = abs(res_fb.custo_otimo - custo_dijk) / res_fb.custo_otimo * 100
    print(f'\n🎯 Gap FB vs Dijkstra: {gap:.4f}% ({"✅ ótimo" if gap < 0.001 else "⚠️ sub-ótimo"})')

## 5. ⚙️ Prim — Árvore Geradora Mínima


In [ ]:
# ⚙️ PARÂMETRO — raiz da MST
ID_RAIZ_MST = 4314902  # Porto Alegre

prim = Prim(grafo)
res_prim = prim.executar(ID_RAIZ_MST)

raiz_nome = grafo.get_vertice(ID_RAIZ_MST)[1]
print(f'MST com raiz em: {raiz_nome}')
print(f'Custo total da MST: {res_prim.custo_mst:.2f} h')
print(f'Arestas ({len(res_prim.arestas_mst)} = N-1 = {grafo.num_vertices()-1}):')
print()
for u, v, peso in sorted(res_prim.arestas_mst, key=lambda x: x[2]):
    nu = grafo.get_vertice(u)[1]
    nv = grafo.get_vertice(v)[1]
    print(f'  {nu:<25} ↔ {nv:<25} {peso:.2f} h')

## 6. ⚙️ Benchmark de Desempenho


In [ ]:
# ⚙️ PARÂMETRO — tamanhos de instância para testar
# Cuidado: adicionar N > 12 no FB travará. FB só roda para N <= 12.
NS_TESTE = [3, 4, 5, 6, 7, 8, 9, 10, 12, 20, 50, 100]

resultados = executar_benchmarks(NS_TESTE)
print('\nBenchmark concluído!')

## 7. Figuras Obrigatórias


In [ ]:
# Gera e exibe a Figura 1 — Grafo com MST
fig1_grafo_mst(grafo, res_prim.arestas_mst, ID_RAIZ_MST)
img = mpimg.imread('../figures/fig1_grafo_mst.png')
plt.figure(figsize=(16, 10))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Figura 2 — BST
fig2_bst(bst, max_nos=13)
img = mpimg.imread('../figures/fig2_bst.png')
plt.figure(figsize=(18, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Figura 3 — Desempenho
fig3_desempenho(resultados)
img = mpimg.imread('../figures/fig3_desempenho.png')
plt.figure(figsize=(15, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Figura 4 — Gap de Otimalidade
fig4_gap(resultados)
img = mpimg.imread('../figures/fig4_gap.png')
plt.figure(figsize=(15, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Figura 5 — Tabela de Estruturas
fig5_tabela_estruturas()
img = mpimg.imread('../figures/fig5_estruturas.png')
plt.figure(figsize=(18, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Escala de Decisão

O projeto avalia as alternativas em quatro níveis, considerando simultaneamente:
**qualidade da solução** (gap vs. ótimo), **custo computacional** (tempo × N),
**adequação da estrutura** e **aplicabilidade prática** ao cenário do RS.


In [ ]:
# ⚙️ PARÂMETRO — ajuste os pesos da escala de decisão para sua análise
# Cada critério recebe nota de 0 a 10

escala = [
    {
        'nivel': '★★★★',
        'solucao': 'Dijkstra (Guloso com heap)',
        'gap_otim': 0.0,
        'nota_qualidade': 10,
        'nota_custo': 10,
        'nota_estrutura': 10,
        'nota_pratica': 10,
        'justificativa': 'Gap=0% (ótimo provado). O(V+E)logV. Viável p/ 478 mun. do RS. BST integrada.'
    },
    {
        'nivel': '★★★ ',
        'solucao': 'Prim — MST completa',
        'gap_otim': 0.0,
        'nota_qualidade': 10,
        'nota_custo': 9,
        'nota_estrutura': 9,
        'nota_pratica': 9,
        'justificativa': 'MST ótima. O(E logV). Resolve cobertura total, não rota individual.'
    },
    {
        'nivel': '★★  ',
        'solucao': 'Força Bruta com poda',
        'gap_otim': 0.0,
        'nota_qualidade': 10,
        'nota_custo': 3,
        'nota_estrutura': 6,
        'nota_pratica': 2,
        'justificativa': 'Ótimo garantido, mas O(N!) inviável acima de N≈12. Só usável como oráculo.'
    },
    {
        'nivel': '★   ',
        'solucao': 'Greedy Ingênuo (sem heap)',
        'gap_otim': 55.0,
        'nota_qualidade': 3,
        'nota_custo': 8,
        'nota_estrutura': 2,
        'nota_pratica': 1,
        'justificativa': 'Rápido, mas gap médio ~55-230%. Cai em becos. Inaceitável p/ emergências.'
    },
]

print(f'{"Nível":<6} {"Algoritmo":<30} {"Gap%":>6} {"Qual":>5} {"Custo":>6} {"Estr":>5} {"Prát":>5}  Justificativa')
print('-' * 110)
for e in escala:
    media = (e['nota_qualidade'] + e['nota_custo'] + e['nota_estrutura'] + e['nota_pratica']) / 4
    print(f"{e['nivel']:<6} {e['solucao']:<30} {e['gap_otim']:>5.1f}%"
          f" {e['nota_qualidade']:>5} {e['nota_custo']:>6} {e['nota_estrutura']:>5} {e['nota_pratica']:>5}  "
          f"{e['justificativa'][:60]}")

print('\n🏆 Recomendação: Dijkstra é a única solução viável para o cenário RS completo (478 municípios).')
print('   A estrutura de dados (heapq + dict de distâncias) é o que garante a otimalidade e a eficiência.')

## 9. ⚙️ Experimento Livre — Crie seu próprio grafo

Crie um grafo sintético de qualquer tamanho e compare os algoritmos:


In [ ]:
# ⚙️ PARÂMETRO — mude N e seed para gerar grafos diferentes
N_EXPERIMENTO = 8   # máx 12 se quiser rodar FB; acima disso FB não executa
SEED = 42           # mude o seed para um grafo diferente

g_exp, orig, dest = gerar_grafo_sintetico(N_EXPERIMENTO, seed=SEED)
print(f'Grafo sintético: {g_exp}')
print(f'Origem:  id={orig}  ({g_exp.get_vertice(orig)[1]})')
print(f'Destino: id={dest}  ({g_exp.get_vertice(dest)[1]})')

# Força Bruta
if N_EXPERIMENTO <= 12:
    res_fb_exp = ForcaBruta(g_exp).encontrar_caminho_minimo(orig, dest)
    print(f'\nFB  → custo={res_fb_exp.custo_otimo:.3f}h  chamadas={res_fb_exp.num_chamadas_rec:,}  tempo={res_fb_exp.tempo_ms:.3f}ms')

# Dijkstra
res_dijk_exp = Dijkstra(g_exp).executar(orig)
custo_d = res_dijk_exp.distancias.get(dest, float('inf'))
caminho_d = res_dijk_exp.caminho_ate(dest)
nomes_d = ' → '.join(g_exp.get_vertice(v)[1] for v in caminho_d)
print(f'Dijk → custo={custo_d:.3f}h  relax={res_dijk_exp.arestas_relaxadas}  tempo={res_dijk_exp.tempo_ms:.3f}ms')
print(f'Rota: {nomes_d}')

if N_EXPERIMENTO <= 12 and res_fb_exp.custo_otimo < float('inf') and custo_d < float('inf'):
    gap = abs(res_fb_exp.custo_otimo - custo_d) / res_fb_exp.custo_otimo * 100
    print(f'\n🎯 Gap: {gap:.4f}% ({"✅ ótimo" if gap < 0.001 else "⚠️ sub-ótimo"})')

---
## Conclusão

Este notebook demonstrou interativamente todos os componentes do sistema:

| Componente | Resultado |
|---|---|
| **BST** | Consulta O(k + log N) aos municípios críticos |
| **Dijkstra** | Gap = 0% (ótimo), O((V+E) log V), escalável para 478 municípios |
| **Prim MST** | Cobertura mínima de todas as rotas de suprimento |
| **Força Bruta** | Oráculo de validação, inviável para N > 12 |

**Conexão ODS 13:** O sistema permite que a Defesa Civil do RS priorize atendimento baseado em dados de risco (deriváveis de satélites Sentinel/GOES-16), alinhando computação eficiente à resposta climática.

*FIAP — Global Solution 2026 | Prof. André Marques*
